# Ingest NYC TLC Yellow Taxi

Download Parquet History to Fabric Lakehouse (2009–2025)

**What this notebook does**

This notebook downloads the NYC TLC Yellow Taxi monthly parquet files from the public CloudFront distribution and stores them in a Microsoft Fabric Lakehouse “Files” area using a Hive-style folder layout:

`Files/TLC_Trip_Record_Data/year=YYYY/month=MM/yellow_tripdata_YYYY-MM.parquet`

It is designed to be:

 - Fast (parallel downloads)
 - Safe to re-run (skips already-downloaded files)
 - Resilient (retry + backoff + timeouts)
 - Verifiable (checks expected months, missing files, and zero-byte files)

**Prerequisites**

 - A Fabric notebook attached to the Lakehouse where you want to store the files.
 - The Lakehouse path for Files is available on the local filesystem as /lakehouse/default/Files/... (default Lakehouse mount).

**Source**

NYC TLC Trip Record Data page: https://www.nyc.gov/site/tlc/about/tlc-trip-record-data.page

In [ ]:
import os
import math
import time
import random
import datetime as dt
from dataclasses import dataclass
from concurrent.futures import ThreadPoolExecutor, as_completed
from typing import List, Tuple, Optional

import requests

## Configuration

Use this to adjust date range, concurrency, and output location.

In [ ]:
# ----------------------------
# User configuration
# ----------------------------

# Source base URL (Yellow Taxi parquet)
BASE_URL = "https://d37ci6vzurychx.cloudfront.net/trip-data"

# Inclusive month range
START_YM = "2009-01"
END_YM   = "2025-11"

# Lakehouse Files destination root (local filesystem mount)
DEST_ROOT = "/lakehouse/default/Files/TLC_Trip_Record_Data"

# Download behavior
MAX_WORKERS = 8           # tune (8–32); depends on capacity/network
CHUNK_SIZE = 8 * 1024 * 1024  # 8 MB
TIMEOUT = (10, 120)       # (connect, read) seconds
MAX_RETRIES = 3
BACKOFF_BASE = 0.75       # seconds
VERIFY_TLS = True         # keep True

# Skip if file exists and has size > 0
SKIP_EXISTING = True

## Helper functions

In [ ]:
def ym_range(start_ym: str, end_ym: str) -> List[Tuple[int, int]]:
    """Generate (year, month) tuples from start_ym to end_ym inclusive."""
    sy, sm = map(int, start_ym.split("-"))
    ey, em = map(int, end_ym.split("-"))
    start = dt.date(sy, sm, 1)
    end = dt.date(ey, em, 1)

    out = []
    cur = start
    while cur <= end:
        out.append((cur.year, cur.month))
        # advance one month
        if cur.month == 12:
            cur = dt.date(cur.year + 1, 1, 1)
        else:
            cur = dt.date(cur.year, cur.month + 1, 1)
    return out

def build_url(year: int, month: int) -> str:
    return f"{BASE_URL}/yellow_tripdata_{year}-{month:02d}.parquet"

def dest_path(year: int, month: int) -> str:
    return os.path.join(
        DEST_ROOT,
        f"year={year}",
        f"month={month:02d}",
        f"yellow_tripdata_{year}-{month:02d}.parquet"
    )

def ensure_parent_dir(path: str) -> None:
    os.makedirs(os.path.dirname(path), exist_ok=True)

def file_ok(path: str) -> bool:
    return os.path.exists(path) and os.path.getsize(path) > 0

def request_session() -> requests.Session:
    s = requests.Session()
    # CloudFront can be sensitive to default python user agents; setting one helps with intermittent 403s.
    s.headers.update({
        "User-Agent": "Mozilla/5.0 (FabricNotebook; +https://learn.microsoft.com/fabric)"
    })
    return s

@dataclass
class DownloadResult:
    year: int
    month: int
    url: str
    dest: str
    status: str              # "ok" | "skipped" | "failed"
    bytes_written: int = 0
    error: Optional[str] = None

def download_one(year: int, month: int, sess: requests.Session) -> DownloadResult:
    url = build_url(year, month)
    dest = dest_path(year, month)

    if SKIP_EXISTING and file_ok(dest):
        return DownloadResult(year, month, url, dest, "skipped", os.path.getsize(dest))

    ensure_parent_dir(dest)

    last_err = None
    for attempt in range(1, MAX_RETRIES + 1):
        try:
            with sess.get(url, stream=True, timeout=TIMEOUT, verify=VERIFY_TLS) as r:
                r.raise_for_status()

                bytes_written = 0
                tmp = dest + ".partial"

                with open(tmp, "wb") as f:
                    for chunk in r.iter_content(chunk_size=CHUNK_SIZE):
                        if chunk:
                            f.write(chunk)
                            bytes_written += len(chunk)

                # atomic-ish replace
                os.replace(tmp, dest)

                if bytes_written == 0:
                    return DownloadResult(year, month, url, dest, "failed", 0, "Downloaded 0 bytes")

                return DownloadResult(year, month, url, dest, "ok", bytes_written)

        except Exception as e:
            last_err = repr(e)
            # cleanup partial
            try:
                if os.path.exists(dest + ".partial"):
                    os.remove(dest + ".partial")
            except Exception:
                pass

            # exponential backoff with jitter
            sleep_s = BACKOFF_BASE * (2 ** (attempt - 1)) + random.random() * 0.25
            time.sleep(min(sleep_s, 30))

    return DownloadResult(year, month, url, dest, "failed", 0, last_err)

## Run parallel download

In [ ]:
targets = ym_range(START_YM, END_YM)
print(f"Expected months: {len(targets)} ({targets[0][0]}-{targets[0][1]:02d} .. {targets[-1][0]}-{targets[-1][1]:02d})")
print(f"Destination root: {DEST_ROOT}")


results: List[DownloadResult] = []

t0 = time.time()
with ThreadPoolExecutor(max_workers=MAX_WORKERS) as ex:
    # One session per worker is okay; but simplest is one per task (requests Session is not guaranteed thread-safe)
    futures = []
    for y, m in targets:
        futures.append(ex.submit(download_one, y, m, request_session()))

    for fut in as_completed(futures):
        res = fut.result()
        results.append(res)

elapsed = time.time() - t0

ok = sum(1 for r in results if r.status == "ok")
sk = sum(1 for r in results if r.status == "skipped")
fail = [r for r in results if r.status == "failed"]

total_bytes = sum(r.bytes_written for r in results if r.status in ("ok", "skipped"))
print(f"Completed in {elapsed:.1f}s | ok={ok} skipped={sk} failed={len(fail)} | total_bytes={total_bytes/1e9:.2f} GB")
if fail:
    print("First failures:")
    for r in fail[:10]:
        print(f"  {r.year}-{r.month:02d} -> {r.error}")

## Optional retry pass for failures

This is useful when there were transient 403/5xx errors.

In [ ]:
if fail:
    print(f"Retrying {len(fail)} failures with a smaller worker pool...")
    retry_targets = [(r.year, r.month) for r in fail]
    results_retry = []

    t1 = time.time()
    with ThreadPoolExecutor(max_workers=max(4, MAX_WORKERS // 2)) as ex:
        futures = [ex.submit(download_one, y, m, request_session()) for y, m in retry_targets]
        for fut in as_completed(futures):
            results_retry.append(fut.result())

    # merge retry results (prefer retry outcome)
    retry_map = {(r.year, r.month): r for r in results_retry}
    merged = []
    for r in results:
        merged.append(retry_map.get((r.year, r.month), r))
    results = merged

    ok = sum(1 for r in results if r.status == "ok")
    sk = sum(1 for r in results if r.status == "skipped")
    fail = [r for r in results if r.status == "failed"]
    print(f"After retry | ok={ok} skipped={sk} failed={len(fail)} | retry_elapsed={time.time()-t1:.1f}s")

## Verify completeness and file health

This checks:
 - all expected months exist
 - no zero-byte files

In [ ]:
expected_set = {(y, m) for y, m in targets}

present = []
zero_bytes = []
for y, m in targets:
    p = dest_path(y, m)
    if os.path.exists(p):
        sz = os.path.getsize(p)
        present.append((y, m, sz))
        if sz == 0:
            zero_bytes.append((y, m, p))
missing = sorted(list(expected_set - {(y, m) for y, m, _ in present}))

print(f"Expected: {len(targets)}")
print(f"Present:  {len(present)}")
print(f"Missing:  {len(missing)}")
print(f"Zero-byte:{len(zero_bytes)}")

if missing:
    print("Missing months (first 50):", missing[:50])
if zero_bytes:
    print("Zero-byte files (first 50):", zero_bytes[:50])

## Notes / troubleshooting

 - If you see intermittent 403 Forbidden, rerun the notebook (or rely on the retry pass). CloudFront sometimes returns transient errors depending on edge/cache.
 - If you want maximum speed, increase MAX_WORKERS (e.g., 32), but only if your capacity/network can handle it.
 - If rerunning after partial success, SKIP_EXISTING=True prevents re-downloading.